In [42]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
import pickle
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [7]:
!pip install scikeras


  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.6 MB 1.5 MB/s eta 0:00:01
   ------------------- -------------------- 0.8/1.6 MB 1.6 MB/s eta 0:00:01
   -------------------------------- ------- 1.3/1.6 MB 1.6 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 1.6 MB/s  0:00:01
Using cached rich-15.0.0-py3-none-any.whl (310 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)

   ----------------- ---------------------- 3/7 [markdown-it-py]
   ---------------------- ----------------- 4/7 [rich]
   ---------------------- ----------------- 4/7 [rich]
   ---------------------- ----------------- 4/7 [rich]
  Attempting uninstall: keras
   --

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.15.0 requires keras<2.16,>=2.15.0, but you have keras 3.14.1 which is incompatible.


In [8]:
from scikeras.wrappers import KerasClassifier

In [10]:
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [11]:
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [13]:
data['Gender'] = LabelEncoder().fit_transform(data["Gender"])

In [21]:
ohe = OneHotEncoder(handle_unknown="ignore")
geo_encoded = ohe.fit_transform(data[["Geography"]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns = ohe.get_feature_names_out(["Geography"]))

data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

In [25]:
X = data.drop('Exited', axis=1)
y = data['Exited']

In [26]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [33]:
scaler = StandardScaler()
x_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

c:\Users\shanz\anaconda3\envs\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [35]:
# Save the encoders and scaler for later use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(LabelEncoder(), file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(ohe, file)

with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

## Define a function to create the model and try different parameters (KerasClassifier)

In [50]:
def create_model(neurons=32, layers=1):
    model= Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1], )))

    for _ in range(layers-1):
        model.add(Dense(neurons, activation='relu'))
    
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    return model

In [51]:
## Create a KerasClassifier for use in scikit-learn

model = KerasClassifier(layers = 1, neurons=32, build_fn= create_model, verbose=1)

In [56]:
param_grid={
    'neurons':[16,32,64,128],
    'layers': [1,2],
    'epochs':[50,100]
}

In [57]:
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=1, cv=3)
grid_result = grid.fit(X_train, y_train)

c:\Users\shanz\anaconda3\envs\venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\shanz\anaconda3\envs\venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\shanz\anaconda3\envs\venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X, y)
c:\Users\shanz\anaconda3\envs\venv\Lib\site-packages\scikeras\wrappers.py:925: UserWarning: ``build_fn`` will be renamed to ``model`` in a future release, at which point use of ``build_fn`` will raise an Error instead.
  X, y = self._initialize(X,

ValueError: 
All the 48 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
48 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\shanz\anaconda3\envs\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\shanz\anaconda3\envs\venv\Lib\site-packages\scikeras\wrappers.py", line 1501, in fit
    super().fit(X=X, y=y, sample_weight=sample_weight, **kwargs)
  File "c:\Users\shanz\anaconda3\envs\venv\Lib\site-packages\scikeras\wrappers.py", line 770, in fit
    self._fit(
  File "c:\Users\shanz\anaconda3\envs\venv\Lib\site-packages\scikeras\wrappers.py", line 925, in _fit
    X, y = self._initialize(X, y)
           ^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Use